Language: **English (default)** | [Polski](lab.ipynb)

# Lab 09 -- SMS classification (spam/ham) with RNN/LSTM/GRU

Companion material to [`README.md`](README.md). A different problem than
the lecture: real text classification (SMS Spam Collection), not
synthetic sequences.

In [1]:
import re  # noqa: F401 - used in the TODO below
import time
from collections import Counter  # noqa: F401 - used in the TODO below

import matplotlib.pyplot as plt  # noqa: F401 - used in the TODO below
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence  # noqa: F401 - used in the TODO below

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


## 1. Data: SMS Spam Collection

In [2]:
import hashlib
import os
import urllib.request
import zipfile

ZIP_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
EXPECTED_SHA256 = "1587ea43e58e82b14ff1f5425c88e17f8496bfcdb67a583dbff9eefaf9963ce3"
CACHE_DIR = "data_cache"
ZIP_PATH = os.path.join(CACHE_DIR, "sms_spam_collection.zip")
TXT_PATH = os.path.join(CACHE_DIR, "SMSSpamCollection")

os.makedirs(CACHE_DIR, exist_ok=True)
if not os.path.exists(TXT_PATH):
    if not os.path.exists(ZIP_PATH):
        urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    with open(ZIP_PATH, "rb") as f:
        actual_sha256 = hashlib.sha256(f.read()).hexdigest()
    assert actual_sha256 == EXPECTED_SHA256, (
        f"SHA-256 mismatch: expected {EXPECTED_SHA256}, got {actual_sha256}"
    )
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extract("SMSSpamCollection", CACHE_DIR)

labels_all, texts_all = [], []
with open(TXT_PATH, encoding="utf-8") as f:
    for line in f:
        label, text = line.rstrip("\n").split("\t", 1)
        labels_all.append(label)
        texts_all.append(text)

print(f"number of messages: {len(texts_all)}")
print("example (ham):", next(t for t, lb in zip(texts_all, labels_all) if lb == "ham")[:60])
print("example (spam):", next(t for t, lb in zip(texts_all, labels_all) if lb == "spam")[:60])

number of messages: 5574
example (ham): Go until jurong point, crazy.. Available only in bugis n gre
example (spam): Free entry in 2 a wkly comp to win FA Cup final tkts 21st Ma


In [3]:
# TODO: count and print the number (and percentage) of messages in each
# class (ham/spam)
raise NotImplementedError("TODO: class distribution")

NotImplementedError: TODO: class distribution

_(Your answer here.)_

## 2. Tokenization

In [4]:
def tokenize(text):
    # TODO: return a list of tokens -- lowercase, words as separate
    # tokens, digits as separate tokens, punctuation as separate (single)
    # tokens. Hint: re.findall with an alternation pattern (a word | digits
    # | a single non-alphanumeric, non-space character)
    raise NotImplementedError("TODO: tokenize")


print(tokenize("Free entry! Win a prize NOW, call 0800-123456"))

NotImplementedError: TODO: tokenize

**Task:** what are the limitations of such simple tokenization (word
forms, punctuation, out-of-vocabulary words)?

_(Your answer here.)_

## 3. Train/val/test split (provided)

In [5]:
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(texts_all), generator=g).tolist()
n = len(texts_all)
n_train, n_val = int(0.7 * n), int(0.15 * n)
train_idx = perm[:n_train]
val_idx = perm[n_train:n_train + n_val]
test_idx = perm[n_train + n_val:]

train_texts_raw = [texts_all[i] for i in train_idx]
train_labels = [1 if labels_all[i] == "spam" else 0 for i in train_idx]
val_texts_raw = [texts_all[i] for i in val_idx]
val_labels = [1 if labels_all[i] == "spam" else 0 for i in val_idx]
test_texts_raw = [texts_all[i] for i in test_idx]
test_labels = [1 if labels_all[i] == "spam" else 0 for i in test_idx]

train_texts = [tokenize(t) for t in train_texts_raw]
val_texts = [tokenize(t) for t in val_texts_raw]
test_texts = [tokenize(t) for t in test_texts_raw]

print(f"train={len(train_texts)}  val={len(val_texts)}  test={len(test_texts)}")
print(f"train spam ratio={sum(train_labels)/len(train_labels):.3f}")

NotImplementedError: TODO: tokenize

## 4. Vocabulary: only from the training set

In [6]:
MIN_FREQ = 2
# TODO:
# 1. count the frequency of every token EXCLUSIVELY in train_texts
#    (collections.Counter)
# 2. build vocab: the list ["<PAD>", "<UNK>"] + tokens with frequency >= MIN_FREQ
# 3. word_to_id: a dict token -> index (position in vocab)
raise NotImplementedError("TODO: building the vocabulary")
PAD_ID, UNK_ID = 0, 1


def encode(tokens, max_len=50):
    # TODO: turn a list of tokens into a list of ids (word_to_id), a token
    # outside the vocabulary -> UNK_ID; truncate to max_len
    raise NotImplementedError("TODO: encode")

NotImplementedError: TODO: building the vocabulary

**Task (code):** verify that a token outside the training vocabulary
does not cause an error, it is simply mapped to `<UNK>`.

In [7]:
fake_token = "zzzznieistniejacyslowo"
# TODO: check that fake_token is NOT in word_to_id, and that
# encode([fake_token]) does not raise an exception and returns UNK_ID
raise NotImplementedError("TODO: UNK verification")

NotImplementedError: TODO: UNK verification

## 5. Variable-length batches (provided)

In [8]:
def make_batch_tensors(encoded_texts, labels_subset):
    lengths = torch.tensor([max(len(e), 1) for e in encoded_texts])
    max_len = lengths.max().item()
    padded = torch.full((len(encoded_texts), max_len), PAD_ID, dtype=torch.long)
    for i, e in enumerate(encoded_texts):
        if len(e) > 0:
            padded[i, :len(e)] = torch.tensor(e, dtype=torch.long)
    return padded, lengths, torch.tensor(labels_subset)


train_encoded = [encode(t) for t in train_texts]
val_encoded = [encode(t) for t in val_texts]
test_encoded = [encode(t) for t in test_texts]

demo_X, demo_lengths, demo_y = make_batch_tensors(train_encoded[:2], train_labels[:2])
print(f"two messages of different length in one batch: shape={tuple(demo_X.shape)}")
print(f"real lengths: {demo_lengths.tolist()}")
shorter_row = demo_X[0] if demo_lengths[0] < demo_lengths[1] else demo_X[1]
print(f"shorter message (padded with PAD_ID={PAD_ID} at the end):\n{shorter_row}")

NameError: name 'train_texts' is not defined

### Training loop (provided, used by every model in this notebook)

In [9]:
def train_eval(model_cls, vocab_size, epochs=5, lr=0.001, batch_size=32, seed=0, **model_kwargs):
    torch.manual_seed(seed)
    model = model_cls(vocab_size, **model_kwargs)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    n = len(train_encoded)

    start = time.time()
    history = {"train_loss": [], "val_loss": []}
    for epoch in range(epochs):
        model.train()
        perm_e = torch.randperm(n)
        total_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm_e[i:i + batch_size].tolist()
            batch_enc = [train_encoded[j] for j in idx]
            batch_lab = [train_labels[j] for j in idx]
            X, lengths, y = make_batch_tensors(batch_enc, batch_lab)
            opt.zero_grad()
            loss = loss_fn(model(X, lengths), y)
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(idx)
        train_loss = total_loss / n
        history["train_loss"].append(train_loss)

        model.eval()
        with torch.no_grad():
            Xv, lv, yv = make_batch_tensors(val_encoded, val_labels)
            val_loss = loss_fn(model(Xv, lv), yv).item()
        history["val_loss"].append(val_loss)
    train_time = time.time() - start

    model.eval()
    with torch.no_grad():
        Xv, lv, yv = make_batch_tensors(val_encoded, val_labels)
        val_acc = (model(Xv, lv).argmax(1) == yv).float().mean().item()

    return model, history, val_acc, train_time

## 6. An order-blind baseline

In [10]:
D_EMBED = 32


class MeanPoolBaseline(nn.Module):
    def __init__(self, vocab_size, d_embed=D_EMBED, n_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_embed, padding_idx=PAD_ID)
        self.fc = nn.Linear(d_embed, n_classes)

    def forward(self, x, lengths):
        # TODO:
        # 1. e = self.embed(x)  (B, T, d_embed)
        # 2. build a mask (x != PAD_ID), to EXCLUDE padding from the average
        # 3. compute the mean of the embeddings ONLY over the real tokens
        #    (use lengths, not the total number of columns T -- otherwise
        #    padding "dilutes" the result for shorter messages)
        # 4. return self.fc(pooled)
        raise NotImplementedError("TODO: MeanPoolBaseline.forward")


baseline_model, baseline_history, baseline_val_acc, baseline_time = train_eval(
    MeanPoolBaseline, len(vocab)  # noqa: F821 - vocab defined in section 4
)
print(f"baseline (mean-pool): val_acc={baseline_val_acc:.3f}  time={baseline_time:.2f}s  "
      f"params={sum(p.numel() for p in baseline_model.parameters())}")

NameError: name 'vocab' is not defined

## 7. An `nn.Embedding` + `nn.GRU` classifier

In [11]:
D_HIDDEN = 32


class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, d_embed=D_EMBED, d_hidden=D_HIDDEN, n_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_embed, padding_idx=PAD_ID)
        self.gru = nn.GRU(d_embed, d_hidden, batch_first=True)
        self.fc = nn.Linear(d_hidden, n_classes)

    def forward(self, x, lengths):
        # TODO:
        # 1. e = self.embed(x)
        # 2. pack e through pack_padded_sequence (batch_first=True,
        #    enforce_sorted=False, lengths.cpu()) BEFORE passing it to
        #    self.gru -- this is the key step that makes the GRU NOT
        #    process the padding
        # 3. self.gru(packed) returns (output, h_n) -- h_n is the state
        #    AFTER THE LAST REAL token of each sequence (not after the
        #    last column of the tensor!)
        # 4. return self.fc(h_n.squeeze(0))
        raise NotImplementedError("TODO: GRUClassifier.forward")

**Verification: the same text must give the same representation
regardless of what it is batched with.**

In [12]:
torch.manual_seed(0)
check_model = GRUClassifier(len(vocab))  # noqa: F821 - vocab defined in section 4
check_model.eval()

single_text = train_encoded[0]
longer_text = max(train_encoded, key=len)

X_alone, len_alone, _ = make_batch_tensors([single_text], [0])
X_batched, len_batched, _ = make_batch_tensors([single_text, longer_text], [0, 0])

with torch.no_grad():
    pred_alone = check_model(X_alone, len_alone)
    pred_batched = check_model(X_batched, len_batched)[0:1]

match = torch.allclose(pred_alone, pred_batched, atol=1e-5)
print(f"same message, same result regardless of padding: {match}")
assert match, (
    "BUG: the representation depends on padding -- check how lengths is "
    "handled in forward()"
)

NameError: name 'vocab' is not defined

## 8. Training the classifier from section 7 (the loop from section 5 -- a provided call)

In [13]:
gru_model, gru_history, gru_val_acc, gru_time = train_eval(
    GRUClassifier, len(vocab)  # noqa: F821 - vocab defined in section 4
)
print(f"GRU: val_acc={gru_val_acc:.3f}  time={gru_time:.2f}s")
print(f"training loss (last epoch): {gru_history['train_loss'][-1]:.4f}")
print(f"validation loss (last epoch): {gru_history['val_loss'][-1]:.4f}")

NameError: name 'vocab' is not defined

## 9. Comparing >=2 configurations

**Task:** which two configurations are you comparing, and why these specifically?

_(Your answer here.)_

In [14]:
# TODO:
# 1. write LSTMClassifier, analogous to GRUClassifier (section 7), but
#    with nn.LSTM -- remember that LSTM returns (output, (h_n, c_n)), not
#    (output, h_n)
# 2. train it via train_eval (the same pattern as for GRU)
# 3. print side by side: val_acc, training time, parameter count for BOTH
raise NotImplementedError("TODO: second configuration and comparison")

NotImplementedError: TODO: second configuration and comparison

## 10. Evaluation on the test set

In [15]:
def macro_f1(preds, targets, n_classes=2):
    # TODO: compute F1 for each class separately (precision, recall, F1
    # from tp/fp/fn), return the AVERAGE (macro-average, not weighted by
    # example count)
    raise NotImplementedError("TODO: macro_f1")


# TODO:
# 1. choose the better model by validation (gru_val_acc vs lstm_val_acc)
# 2. compute predictions ON THE TEST SET (the first time in this notebook)
# 3. compute test_acc, test_macro_f1 (use macro_f1 above)
# 4. build and draw a 2x2 confusion matrix
raise NotImplementedError("TODO: test evaluation")

NotImplementedError: TODO: test evaluation

## 11. Diagnostic task

The following code **looks** correct -- it raises no exception, it
trains -- but it has a bug related to padding handling.

In [16]:
class BuggyClassifier(nn.Module):
    def __init__(self, vocab_size, d_embed=D_EMBED, d_hidden=D_HIDDEN, n_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_embed, padding_idx=PAD_ID)
        self.gru = nn.GRU(d_embed, d_hidden, batch_first=True)
        self.fc = nn.Linear(d_hidden, n_classes)

    def forward(self, x, lengths):
        e = self.embed(x)
        out, _ = self.gru(e)  # MISSING pack_padded_sequence!
        last_step = out[:, -1, :]  # always the LAST COLUMN of the tensor
        return self.fc(last_step)


torch.manual_seed(0)
buggy_model = BuggyClassifier(len(vocab))  # noqa: F821 - vocab defined in section 4
buggy_model.eval()

X_alone_b, len_alone_b, _ = make_batch_tensors([single_text], [0])
X_batched_b, len_batched_b, _ = make_batch_tensors([single_text, longer_text], [0, 0])

with torch.no_grad():
    pred_alone_b = buggy_model(X_alone_b, len_alone_b)
    pred_batched_b = buggy_model(X_batched_b, len_batched_b)[0:1]

print("same text, BuggyClassifier, batched alone vs. with a longer sequence:")
print("  alone:   ", pred_alone_b.flatten().tolist())
print("  in batch:", pred_batched_b.flatten().tolist())
print("  identical?", torch.allclose(pred_alone_b, pred_batched_b, atol=1e-5))

NameError: name 'vocab' is not defined

**Task:** diagnose the bug (a specific line), explain when it shows up
(check for batch_size=1 separately), fix the code, verify on sequences of
different lengths.

In [17]:
# TODO:
# 1. write the diagnosis (as a comment): which line is the source of the
#    bug, and why
# 2. write FixedClassifier -- a corrected version of BuggyClassifier (use
#    pack_padded_sequence, as in section 7)
# 3. verify analogously to section 7 (the same text, alone vs. in a batch)
raise NotImplementedError("TODO: diagnosis and fix")

NotImplementedError: TODO: diagnosis and fix

## 12. Error analysis

In [18]:
# TODO:
# 1. find the indices of misclassified test examples
# 2. print 5-8 of them (text + true/predicted label)
# 3. count how many errors are ham->spam and how many are spam->ham
# 4. compare the average length (in tokens) of correctly and incorrectly
#    classified messages
raise NotImplementedError("TODO: error analysis")

NotImplementedError: TODO: error analysis

_(Your answer here.)_

## 13. Conclusions

**1. By how much does the sequential model beat the baseline? What does that say about how much spam classification requires understanding order, compared to the lecture's task?**

_(Your answer here.)_

**2. Which configuration from section 9 did better? Is the difference convincing?**

_(Your answer here.)_

**3. What is at least one limitation of this experiment?**

_(Your answer here.)_